# L13b: Interpretable Allocation and Model-Predictive Rebalancing
Previously, we chose the share counts of a portfolio once, by maximizing a utility function under a budget. How do we manage that portfolio every day, and how do we judge it when the real market gives us only one path?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Pose the allocation problem with trading limits:__ Write the maximum-utility problem over share counts with its budget and trading limits. Explain how the score of a stock combines its expected growth rate with the market state.
> * __Explain when a model-predictive controller re-plans:__ Describe the forecast band and the three triggers that start a re-plan. Read a real-data backtest that compares selection rules and weighting rules.
> * __Evaluate a trading method beyond the historical record:__ Explain why a backtest is only one path. Describe how synthetic markets test the same method in typical and stressed conditions.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * Each rebalance maximizes a utility of the stock scores within a budget and four trading limits. The score has the sign of the stock's expected growth rate, and the market state changes only its size.
> * The method re-plans after 21 trading days, when the wealth leaves the 95% forecast band, or when it falls 8% below its peak. Between re-plans, the method does not trade.
> * A backtest is one historical path. A market generator fitted to the same stocks gives many typical and stressed markets for testing the same method.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The four utilities, their solutions under the budget alone, and the three benchmarks.
> * The forecast model, the market hedge, and the trading costs.
> * The ten selection rules, and how the typical and stress sets are chosen.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Run the INFORMS portfolio method on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we calibrate the method on 2014 to 2024 for 373 S&P 500 stocks and trade 2025 and 2026. We run one portfolio day by day, then change the weighting rule, the selection rule, the hedge, and the basket on the same path.

> [▶ Run the INFORMS portfolio method on synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw typical markets and two kinds of sell-off from a generator fitted to the same 373 stocks. We run the method on each market and compare the portfolios with `SPY` and an equal-weight portfolio.

___

## Live Portfolio Management
Live portfolio management couples three problems: allocation (what to hold), rebalancing (when to trade), and evaluation beyond the historical record. We'll treat the three problems as one system. We call the portfolio that this system manages the __book__.

This lecture follows J. D. Varner, "Interpretable Allocation and Model-Predictive Rebalancing of Equity Portfolios, Tested on Real and Synthetic Markets", INFORMS Annual Meeting 2026, [arXiv:2611.03022](https://arxiv.org/abs/2611.03022). The study's code, results, and slides are in a [public repository](https://github.com/varnerlab/INFORMS-Annual-Meeting-2026-Varnerlab).

The study uses daily data for 373 S&P 500 stocks. It calibrates on 2014 to 2024 and trades the 420 trading days from January 2025 to September 2026. Every decision uses only data available on the day of the decision.
___

## Allocation
What should the book hold? A __selection rule__ picks a basket $\mathcal{N}$ of $K$ stocks, or __names__, from the universe. A __weighting rule__ then chooses the number of shares $n_{i}\geq 0$ of each stock $i$ in the basket. The method gives each stock a score.

### The score γ: expected growth, tilted by the market
After every close, the score of stock $i$ is given by:

$$
\boxed{\gamma_{i}=\tanh\left(\frac{\alpha_{i}+\beta_{i}\,\tilde{g}_{M}}{\max\left(|\beta_{i}|,\,0.1\right)^{\xi}}\right)}
$$

where $\alpha_{i}$ and $\beta_{i}$ are the single index model parameters of stock $i$ against `SPY`. The market growth rate $\tilde{g}_{M}$ is the 252-day exponential moving average (EMA) of the daily growth rate of `SPY` (units: 1/yr). The numerator is the __expected growth rate__ of stock $i$, and the denominator is the __β penalty__.

The penalty depends on the __market state__ $\xi$, which compares the 21-day and the 63-day EMA of the `SPY` close:

$$
\xi=-G\left(\frac{\text{EMA}_{21}}{\text{EMA}_{63}}-1\right)
$$

where the gain $G=21.23$ keeps $|\xi|\leq 1$ on 99% of the days from 2014 to 2024. The market state is positive in a falling market, where $\text{EMA}_{21}<\text{EMA}_{63}$, and negative in a rising one.

Let's look at the market state over the trading window:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-MarketState.svg" width="900" style="max-width:100%; height:auto;" alt="Two strips share a date axis from January 2025 to September 2026. The top strip shows the SPY close in gold with its 21-day exponential moving average in black and its 63-day exponential moving average in blue, each divided by the first SPY close. It rises from 1.0 to about 1.3, with a dip to about 0.9 in April 2025. The bottom strip shows the market state, which stays between minus 0.8 and 0 for most of the window and rises above zero twice, to about 0.9 in April 2025 and to about 0.4 in March 2026. Both strips shade the days on which the market state is positive, from March to May 2025 and from March to April 2026. Four open rings mark the days on which the two moving averages cross, and a dotted line runs from each ring down to the point where the market state crosses zero."/>
    </center>
</div>

__The market state, January 2025 to September 2026.__ The top strip shows `SPY` and its two EMAs, each divided by the first `SPY` close. The bottom strip shows $\xi$. The market state was positive on the shaded days, 75 of the 420 trading days.

__How do we read the score?__ The score has the sign of the expected growth rate, because the $\beta$ penalty is always positive. In a falling market ($\xi>0$), the penalty shrinks the scores of stocks with $|\beta_{i}|>1$ toward zero and favors stocks with a low $\beta$. In a rising market ($\xi<0$), the score favors stocks with a high $\beta$.

The hyperbolic tangent keeps the score between −1 and 1. The absolute value and the floor of 0.1 extend the preference weight we used previously to stocks with $\beta_{i}\leq 0$.


### Maximize utility subject to trading limits
Let $S_{i}$ denote the close price of stock $i$ (units: USD/share) and $W$ the wealth of the book (units: USD). The weight of stock $i$ is $w_{i}=n_{i}S_{i}/W$.

Each rebalance chooses the shares $n=(n_{1},\dots,n_{K})$ to solve:

$$
\boxed{\begin{aligned}
\operatorname*{maximize}_{n\,\geq\,0}\quad & U(n;\gamma) &&\quad\text{(utility)}\\
\text{subject to}\quad & \sum_{i\in\mathcal{N}}n_{i}S_{i}\leq W &&\quad\text{(budget)}\\
& w_{i}\leq\bar{w} &&\quad\text{(single-name cap)}\\
& \sqrt{w^{\top}\Sigma_{g}\,w}\leq\sigma_{\text{SPY}} &&\quad\text{(volatility cap)}\\
& \bar{c}\,\lVert n-n^{\text{prev}}\rVert_{1}\leq B &&\quad\text{(turnover budget)}\\
& \beta^{\top}w\leq\bar{\beta} &&\quad\text{(}\beta\text{ cap, hedge on)}
\end{aligned}}
$$

where $n^{\text{prev}}$ holds the shares before the rebalance. Wealth that the budget leaves unspent stays in cash.

The four trading limits are:

| Limit | What it does | Setting in the study |
|:--|:--|:--|
| Single-name cap $\bar{w}$ | Each stock is at most the fraction $\bar{w}$ of the wealth. | $\max(0.10,\,1.5/K)$, which is 11.5% for 13 stocks |
| Volatility cap $\sigma_{\text{SPY}}$ | The book is no more volatile than `SPY`, where $\Sigma_{g}$ is the covariance of the daily growth rates. | 17.3% a year |
| Turnover budget $B$ | A rebalance spends at most $B$ on the half spread $\bar{c}$ per share traded. | $0.10\,W$, with a half spread of 5 basis points of the mean close price |
| $\beta$ cap $\bar{\beta}$ | With the __market hedge__ on, the book's $\beta$ to `SPY` is at most $\bar{\beta}$. | $\max(0,\,1-\xi)$ |

With the hedge on, the book can also hold up to five stocks with $\beta_{i}\leq 0$.

The study compares four utilities of the stocks with $\gamma_{i}>0$:

| Utility | $U(n;\gamma)$ | Solution with the budget alone |
|:--|:--|:--|
| Cobb–Douglas | $\sum_{i}\gamma_{i}\ln n_{i}$ | $w_{i}\propto\gamma_{i}$ |
| CES on shares | $\left(\sum_{i}\gamma_{i}\,n_{i}^{\rho}\right)^{1/\rho}$ | $n_{i}\propto\left(\gamma_{i}/S_{i}\right)^{\eta}$ |
| CES on weights | $\left(\sum_{i}\gamma_{i}\,w_{i}^{\rho}\right)^{1/\rho}$ | $w_{i}\propto\gamma_{i}^{\eta}$ |
| Stone–Geary | $\sum_{i}\gamma_{i}\ln\left(n_{i}-b_{i}\right)$ | $\left(n_{i}-b_{i}\right)S_{i}\propto\gamma_{i}$ |

where $\rho=1-1/\eta$ for the elasticity $\eta=0.5$, and the Stone–Geary baseline $b_{i}$ is the number of shares worth 1% of the wealth. With the trading limits, a numerical solver finds the maximum.

The study also runs three benchmarks under the same limits: minimum variance, the tangency portfolio, and a 130/30 long–short book that shorts stocks with $\gamma_{i}<0$. The four utilities and the three benchmarks are the seven weighting rules.
___

## Rebalancing
When should the book trade? Trading every day costs money. A fixed calendar ignores what the market does between trades.

### Rebalancing triggers: horizon, forecast band, drawdown
The method uses __model-predictive control__. The controller forecasts the wealth of the book and measures the real wealth after each close. When the measurement departs from the forecast, the controller makes a new plan, called a __re-plan__.

At each re-plan, the controller simulates 1,000 price paths over the next 21 trading days with a multi-asset geometric Brownian motion. The paths use the same expected growth rates and covariance $\Sigma_{g}$ as the allocation. Let $\mu_{W}(\tau)$ and $\sigma_{W}(\tau)$ denote the mean and the standard deviation of the forecast wealth $\tau$ trading days after the re-plan.

The 95% __forecast band__ for the wealth $W$ is given by:

$$
\mu_{W}(\tau)-1.96\,\sigma_{W}(\tau)\;\leq\;W\;\leq\;\mu_{W}(\tau)+1.96\,\sigma_{W}(\tau)
$$

After each close, the controller checks three triggers in order and re-plans on the first trigger that holds:

* __Drawdown stop:__ The wealth is more than 8% below its running peak. The re-plan resets the peak to the current wealth.
* __Horizon:__ The plan has run for 21 trading days, where the forecast ends. The next forecast starts from the measured wealth.
* __Band exit:__ The wealth has left the 95% band. The book has gained or lost more than the model expected.

Each re-plan solves the same allocation problem from the shares held, with that day's estimates. Between re-plans, the book does not trade.

Let's look at one episode of the real-data backtest:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-ForecastBand.svg" width="860" style="max-width:100%; height:auto;" alt="The wealth of one book divided by its starting wealth, in red, over 50 trading days from October 10 to December 22, 2025. It rises from 1.01 to about 1.15. Three blue fans of forecast bands, each with a 68 percent, a 95 percent, and a 99 percent band, start at October 10, November 10, and November 20 and widen with time. Gold rings mark three re-plans. The first, on November 10, is labeled 21 days passed. The second, on November 20, is labeled left the 95 percent band, 2.1 standard deviations below the mean. The third, on December 22, is labeled 21 days passed."/>
    </center>
</div>

__One episode, October 10 to December 22, 2025.__ The red line is the wealth of the book divided by its starting wealth $W_{0}$. The blue bands are the 68%, 95%, and 99% bands of three forecasts, and each gold ring is a re-plan.

This book holds 13 hand-picked stocks, with the Cobb–Douglas utility and the hedge off. The first and third re-plans came at the horizon. The second came when the wealth fell 2.1 standard deviations below the forecast mean.

From January 2025 to September 2026, this book re-planned 27 times: 17 at the horizon, 7 at a band exit, and 3 at a drawdown stop.

__Do the triggers beat a fixed schedule?__ On a fixed 21-day schedule, the ending wealth was the same. The triggers took about a point off the worst drawdown, on real prices and in simulated sell-offs.

__What does a trade cost?__ Each trade fills at the next open and pays a half spread of 5 basis points plus a market-impact cost. There is no commission.


### Stock selection matters more than the weighting rule
We ran every pair of ten selection rules and seven weighting rules on the real data, with the hedge on and off. The selection rules come in three groups:

* __33 stocks, 3 per sector:__ In each of the 11 sectors, the rule takes the 3 highest scores, the top 3 stocks of a bandit, or 3 stocks at random. The bandit learns which stocks in a sector grew faster than `SPY` over the trailing year.
* __13 stocks:__ Three rules take the 13 highest scores: from the whole universe, from the 100 most traded stocks, or skipping a stock correlated above 0.7 with one already picked. Two random rules draw 13 stocks from the same two pools.
* __Hand-picked:__ The book holds 13 stocks chosen by hand without 2025 data and never re-picks them. A second list adds WMT and PG.

Let's look at the terminal wealth of every pair:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-RealData-Selection.svg" width="880" style="max-width:100%; height:auto;" alt="A dot plot of terminal wealth divided by initial wealth, from 1.0 to 1.7, with one row for each of ten selection rules in three groups. The group of 33 stocks, 3 per sector, has the rows gamma per sector, bandit per sector, and random per sector, and its dots lie between about 1.07 and 1.29. The group of 13 stocks has the rows global gamma, correlation-aware gamma, liquid then gamma, random from the liquid 100, and random from the whole universe, and its dots lie between about 0.99 and 1.59. The hand-picked group has the rows hand-picked 13 and hand-picked 13 plus WMT and PG, and its dots lie between about 1.16 and 1.70. Each row holds seven filled dots for the weighting rules with the hedge off and seven open dots with the hedge on. A dashed gold line marks SPY bought and held at 1.32, and a dotted line marks the equal-weight universe at 1.30."/>
    </center>
</div>

__Terminal wealth on real data.__ Each row is a selection rule, and each dot is a weighting rule. Filled dots have the hedge off, and open dots have it on. The dashed line is `SPY` bought and held, and the dotted line is an equal-weight portfolio of all 373 stocks.

__What do we see?__ The dots in a row are close together, and the rows are far apart. With the hedge off, the seven weighting rules on the 33 stocks picked by score ended between 1.09 and 1.23 times the initial wealth. Different baskets ended between 1.03 and 1.70, and the hand-picked list ended highest.

> __Example:__
>
> [▶ Run the INFORMS portfolio method on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we run one book day by day. We then change the weighting rule, the selection rule, the hedge, and the basket on the same path.

One path cannot tell us whether a rule was good or lucky.
___

## Market Generator
How do we judge the method beyond the historical record? Real data gives one path of prices, with one broad sell-off in the spring of 2025. A market generator gives many synthetic markets for the same stocks.

### JumpHMM: a market generator with fat tails
Previously, we built a market generator from jump hidden Markov models (JumpHMM).

The growth rate of synthetic stock $i$ in sector $s$ on trading day $t$ is given by:

$$
g_{i}(t)=\underbrace{\alpha_{i}}_{\text{trend}}+\underbrace{\beta_{i}(t)\,g_{m}(t)}_{\text{market}}+\underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}}+\underbrace{\psi_{i}\,\tilde{\varepsilon}_{i}(t)}_{\text{own}}
$$

where the market draw $g_{m}(t)$, the sector draw $f_{s}(t)$, and the own draw $\tilde{\varepsilon}_{i}(t)$ each come from a JumpHMM. The loadings $\delta_{i}$ and $\psi_{i}$ scale the sector and own draws, and each $\beta_{i}(t)$ drifts around the stock's estimate.

The generator is fitted to the same 373 stocks on 2014 to 2024. It reproduces the fat tails of real growth rates, but its volatility clustering fades after a day. The controller still forecasts with a geometric Brownian motion, which has no fat tails.

### Simulated sell-offs: every median ends above SPY
On each synthetic market, the method calibrates on 2014 to 2024 and trades January 2025 to September 2026 with the real settings. We draw 10,000 markets and build three sets from them:

* __Typical:__ This set holds 600 markets drawn at random. The median `SPY` drawdown is 17.6%.
* __Burst stress:__ This set holds the 200 deepest `SPY` drawdowns among markets with a __jump burst__, months of large moves of both signs. The median `SPY` drawdown is 36.4%.
* __Bear stress:__ This set holds the 200 deepest `SPY` drawdowns among markets without a burst, where prices fall more slowly. The median `SPY` drawdown is 40.6%.

Let's look at the terminal wealth in each set:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-Synthetic-Sets.svg" width="940" style="max-width:100%; height:auto;" alt="Three dot plots side by side, titled typical, burst stress, and bear stress, share the ten selection rules of the real-data figure as rows. Each dot is the median terminal wealth divided by initial wealth across the futures of a set, for one weighting rule, with filled dots for the hedge off and open dots for the hedge on. In the typical panel the dots lie between about 1.09 and 1.24, around the dashed SPY line at 1.19 and mostly below the dotted equal-weight line at 1.23. In the burst stress panel the dots lie between about 0.92 and 1.03, all to the right of the SPY line at 0.86 and the equal-weight line at 0.92. In the bear stress panel the dots lie between about 0.77 and 1.00, all to the right of the SPY line at 0.72 and the equal-weight line at 0.75."/>
    </center>
</div>

__Terminal wealth in synthetic markets.__ The rows, dots, and lines are as in the real-data figure. Each dot is the median across the markets of a set.

__What do we see?__ In both stress sets, every median ended above `SPY` and above the equal-weight portfolio. The random selection rules did as well. The margin over `SPY` came from the allocation, not from the timing of the re-plans.

__Does selection by score help?__ In typical markets with the hedge off, 13 stocks picked by score beat 13 random stocks in 62 to 71% of the markets. In the bear stress set, random picks did as well or better for most weighting rules. The generator keeps each stock's past trend, which favors selection by score more than real prices did.

__Does the hedge pay?__ The hedge reduced terminal wealth on real prices, but it added to terminal wealth in the bear stress set.

> __Example:__
>
> [▶ Run the INFORMS portfolio method on synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we run the method on markets from each set and compare the books with `SPY` and an equal-weight portfolio.

The example uses a few markets in each set, and its counts are rougher than the study's.
___

## Summary
In this lecture, we followed one system for managing a stock portfolio every day, from the allocation problem to its test on real and synthetic markets. We posed the allocation with trading limits, let a model-predictive controller decide when to re-plan, and judged the method on one real path and on many synthetic markets.

> __Key Takeaways:__
>
> * __In allocation, stock selection mattered more than the weighting rule:__ We maximized a utility of each stock's score within a budget and trading limits. On real data, seven weighting rules on one basket ended close together, while different baskets ended far apart.
> * __In rebalancing, re-plans came at the horizon, a band exit, or a drawdown stop:__ We re-planned when the wealth left the forecast band, fell 8% from its peak, or reached the 21-day horizon. The triggers gave the same ending wealth as a fixed schedule and took about a point off the worst drawdown.
> * __In both sell-off sets, every median ended above SPY:__ We tested the method on typical and stressed markets from a jump hidden Markov model generator fitted to the same stocks. The generator reproduces fat tails, but it also keeps each stock's past trend, which favors selection by score.

The study treats allocation, rebalancing, and evaluation as one system that we can inspect at every step. A paper-trading version of this system is running now.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___